# Results

Every table and figure of the evaluation protocol, rebuilt from the run records in `runs/` (pulled from the private Hub repo).

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from farmvlm import report
from farmvlm.metrics import risk_coverage
RUNS = Path('../runs')
print(report.latex_ablation(report.table_ablation(RUNS)))

In [ ]:
for r in report.table_fold0(RUNS): print(r)

In [ ]:
for r in report.table_hedging(RUNS): print(r)

In [ ]:
for name in ('baseline', 'blip-lora', 'served-rgb-eval', 'served-ndvi-eval'):
    rs = [x for x in report.rows(RUNS / name / 'f0.jsonl') if x['family'] != 'caption']
    if not rs: continue
    conf = np.array([x['conf'] for x in rs]); ok = (report.credits(rs) == 1).astype(float)
    cov, acc, aurc = risk_coverage(conf, ok)
    plt.plot(cov, acc, label=f'{name} (AURC {aurc:.3f})')
plt.xlabel('coverage (answer only the most confident share)'); plt.ylabel('accuracy'); plt.legend(); plt.title('Selective answering'); plt.show()

In [ ]:
rob = report.table_robustness(RUNS)
for name, drops in rob.items():
    print(name, {k: round(v, 3) for k, v in drops.items()})

In [ ]:
print('leakage (seen-field vs unseen-field accuracy, 95% CI):', report.table_leakage(RUNS))

In [ ]:
p = RUNS / 'heatmaps' / 'pointing.json'
if p.exists(): print(json.loads(p.read_text())['summary'])

In [ ]:
for run in sorted(RUNS.glob('blip-*-f0')) + [RUNS / 'served-rgb', RUNS / 'served-ndvi']:
    m = [json.loads(l) for l in (run / 'metrics.jsonl').read_text().splitlines()] if (run / 'metrics.jsonl').exists() else []
    if m: plt.plot([x['pairs'] for x in m], [x['val'] for x in m], marker='o', label=run.name)
plt.xlabel('pairs seen'); plt.ylabel('inner-validation score'); plt.legend(); plt.title('Training curves'); plt.show()